# 4-тарау. PyTorch-та конволюциялық желі

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 4.1. Жоспар

## 4.2. Деректерді жүктеу

In [ ]:
import urllib.request
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

URL = ("https://storage.googleapis.com/tensorflow/"
       "tf-keras-datasets/mnist.npz")
urllib.request.urlretrieve(URL, "mnist.npz")
d = np.load("mnist.npz")

def to_tensors(x, y):
    x = torch.tensor(x, dtype=torch.float32).unsqueeze(1) / 255
    return x, torch.tensor(y, dtype=torch.long)

X_train, y_train = to_tensors(d["x_train"], d["y_train"])
X_test, y_test = to_tensors(d["x_test"], d["y_test"])
print(X_train.shape, X_test.shape)
print(y_train[:10])

## 4.3. Модель

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 16×28×28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 16×14×14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 32×14×14
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 32×7×7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                 # 1568
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, 10),                            # 10 цифр
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN()
print("Параметрлер:", sum(p.numel() for p in model.parameters()))

In [ ]:
x = torch.zeros(1, 1, 28, 28)
for layer in list(model.features) + list(model.classifier):
    x = layer(x)
    print(f"{layer.__class__.__name__:10s} {tuple(x.shape)}")

## 4.4. Оқыту және бағалау

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
model = CNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
train_dl = DataLoader(TensorDataset(X_train, y_train),
                      batch_size=64, shuffle=True)
test_dl = DataLoader(TensorDataset(X_test, y_test), batch_size=1000)

def train_epoch(model, dl):
    model.train()
    total = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    return total / len(dl.dataset)       # орташа шығын

@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    correct = 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()
    return correct / len(dl.dataset)     # дәлдік

for epoch in range(1, 4):
    loss = train_epoch(model, train_dl)
    acc = evaluate(model, test_dl)
    print(f"{epoch}-дәуір: шығын={loss:.4f}, тест дәлдігі={acc:.4f}")

## 4.5. Модель қайда қателеседі?

In [ ]:
from collections import Counter

model.eval()
with torch.no_grad():
    preds = torch.cat([model(xb.to(device)).argmax(1).cpu()
                       for xb, _ in test_dl])
wrong = (preds != y_test).nonzero().squeeze(1)
print("Қате саны:", len(wrong))
pairs = Counter(zip(y_test[wrong].tolist(), preds[wrong].tolist()))
for (true, pred), n in pairs.most_common(5):
    print(f"шын {true} → болжам {pred}: {n} рет")

## 4.6. Модельді сақтау

In [ ]:
torch.save({"model": model.state_dict(),
            "optimizer": opt.state_dict(),
            "epoch": 3}, "checkpoint.pt")

ckpt = torch.load("checkpoint.pt")
model2 = CNN().to(device)
model2.load_state_dict(ckpt["model"])
print(ckpt["epoch"], f"{evaluate(model2, test_dl):.4f}")

**Міндетті емес ұяшық.** Бұл код қосымша бағдарламаны, интернеттегі қызметті немесе өз файлыңызды қажет етеді (толығырақ кітапта). Іске қосу үшін ұяшықтың бірінші жолын (`%%script echo ...`) өшіріңіз.

In [ ]:
%%script echo "Міндетті емес ұяшық: іске қосу үшін осы бірінші жолды өшіріңіз."
from PIL import Image

img = Image.open("my_digit.png").convert("L").resize((28, 28))
x = torch.tensor(np.array(img), dtype=torch.float32) / 255
if x.mean() > 0.5:          # ақ фондағы қара цифр болса,
    x = 1 - x               # түстерді аударамыз
with torch.no_grad():
    probs = torch.softmax(model(x.view(1, 1, 28, 28).to(device)), dim=1)
print("Болжам:", probs.argmax().item(),
      f"сенімділік: {probs.max().item():.2f}")